---
title: A Threads Virality Engine That Learns From Post #10, with TabPFN 3.5
description: Predict a post's views before it goes out, find which levers (time, media, length, hook, hashtags, author) move reach, rewrite drafts under the account's own limits, and watch the ranking become useful within the first dozen posts.
icon: bolt
cookbookTags:
- regression
- causality
- text
- agent
- cold-start
authors:
- name: jinseriouspark
---
# A Threads virality engine that learns from post #10

A new social account has no history. Every post is a guess about timing, length, hooks and whether the model-written draft is better than yours. Conventional models need hundreds of rows before they say anything; by then the account has already spent its first month guessing.

TabPFN 3.5 changes the order of operations. It fits in-context in seconds, reads the post text as text, and is built for the first fifty rows rather than the first fifty thousand. This cookbook puts it in a loop:

1. **Score** a draft before posting: predicted views, an uncertainty band, and the closest past posts.
2. **Explain** which levers move reach on *this* account: posting hour, media, replies, length, hooks, hashtags, links, topic, and human-vs-LLM authorship, each with a cluster-bootstrap interval.
3. **Rewrite** under the account's own limits and let TabPFN rank the variants.
4. **Learn**: a learning curve over the timeline shows where the ranking becomes trustworthy.

Everything runs offline on a gradient-boosted stand-in if `TABPFN_TOKEN` is not set. The point of the notebook is the loop; the point of the token is the model.

In [ ]:
%pip install -q "adlift[client] @ git+https://github.com/jinseriouspark/tabpfn_for_ads" tabulate

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ.setdefault("TABPFN_TOKEN", userdata.get("TABPFN_TOKEN") or "")
except Exception:
    pass

from adlift.model import resolve_backend
BACKEND = resolve_backend("auto")
print("model backend:", BACKEND)

## A timeline with known lever effects

Real view counts for someone else's posts are not available through the Threads API, and an account that started today has no history yet. The generator below writes one account's timeline with every lever's effect planted on the log of views, LLM adoption rising over the period, a growing audience, heavy-tailed views with occasional viral runs, and **both potential texts** per post so the author effect is known exactly.

To run on your own account instead: `adlift threads fetch --token ...` pulls posts and insights through the API into a CSV, `adlift threads template` writes a sheet to label `author` (human / llm) and `topic`, and `adlift threads label` merges it back. Then pass the CSV wherever `timeline` is used below.

In [ ]:
import pandas as pd
from adlift.datasets.synth_threads import make_threads_account, true_log_ate
from adlift.schema import THREADS_SPEC

timeline = make_threads_account(n_weeks=16, seed=7)
print(f"{len(timeline)} posts over {timeline.week.nunique()} weeks; median views {int(timeline.views.median())}, p90 {int(timeline.views.quantile(0.9))}")
print(f"planted author effect: {true_log_ate(timeline, 'total') * 100:+.1f}% views (total), {true_log_ate(timeline, 'direct') * 100:+.1f}% (direct)")
with pd.option_context("display.max_colwidth", 80):
    display(timeline[["week", "posted_hour", "has_media", "topic", "author", "views", "text"]].head(5))

## Score a draft before it goes out

`CopyCoach` loads the timeline as context (with hosted TabPFN this uses `fit_with_cache`, so every later prediction skips the forward pass). A `PostDraft` carries the text and the posting context you control.

In [ ]:
from adlift.llm import get_llm
from adlift.loop import CopyCoach, PostDraft
from adlift.model import CTRModel

coach = CopyCoach(timeline, spec=THREADS_SPEC, model=CTRModel(backend=BACKEND, spec=THREADS_SPEC), llm=get_llm()).fit()
print(f"fitted on {len(timeline)} posts in {coach.fit_seconds:.2f}s ({coach.model.backend})")

draft = PostDraft(
    text="Excited to share how Atlas is transforming the way founders approach ticket backlogs! 🚀 #ai #startup #growth",
    author="llm",
    context={"posted_hour": 9, "weekday": "Sat", "has_media": 0, "is_reply": 0, "topic": "product_launch",
             "week_index": int(timeline.week_index.max())},
)
[scored] = coach.score([draft])
print(f"predicted views {scored.predicted:,.0f}  band [{scored.low:,.0f}, {scored.high:,.0f}]")
coach.precedents(draft, k=5)

## Which single change helps this post most

`suggest_for_post` re-scores the draft under each alternative setting of one lever and ranks the changes. Text levers are evaluated as attribute flips, so "open with a question" is a prompt for the rewrite step, not a rewrite itself.

In [ ]:
from adlift.levers import suggest_for_post

suggestions = suggest_for_post(
    timeline, draft.to_row(THREADS_SPEC), spec=THREADS_SPEC,
    model_factory=lambda: CTRModel(backend=BACKEND, spec=THREADS_SPEC), top=6,
)
print(f"as written: {suggestions.attrs['as_is']:,.0f} views")
suggestions

## What moves reach on this account

`lever_analysis` does the same for every post at once: each lever's effect relative to what the account usually does, with a cluster bootstrap over weeks. `support` says how many posts were actually observed at the alternative; a lever with no support is an extrapolation and the table says so.

In [ ]:
from adlift.levers import lever_analysis

levers = lever_analysis(
    timeline, spec=THREADS_SPEC, model_factory=lambda: CTRModel(backend=BACKEND, spec=THREADS_SPEC), n_boot=200,
)
show = levers[["lever", "from", "to", "effect", "ratio", "ci_low", "ci_high", "support", "significant"]].copy()
show["ratio"] = (show["ratio"] * 100).round(1).astype(str) + "%"
show.round(1).head(14)

## Rewrite under the account's own limits

The constraints come from the account's top-quartile posts: how long the winners run, whether they carry a number, how many hashtags they allow. The language model rewrites inside them; TabPFN ranks the results. Set `ANTHROPIC_API_KEY` to use Claude; otherwise a deterministic stub stands in.

In [ ]:
result = coach.revise(draft, n_variants=6)
print(f"draft {result.draft.predicted:,.0f} -> best {result.best.predicted:,.0f} views  (scored in {result.score_seconds:.2f}s)")
print(f"constraints: <= {result.constraints.max_words} words, max hashtags {result.constraints.max_hashtags}, keep number {result.constraints.keep_numeric_claim}")
pd.DataFrame([v.to_dict() for v in result.variants])[["rank", "predicted", "lift_vs_draft", "word_count", "headline"]]

## Is LLM copy actually worse on this account?

The author is one lever among many, but it is the one people argue about, so it gets the full treatment: a T-learner on the log scale (read as a ratio) with an interval from a bootstrap that refits both arms, an S-learner as cross-check, a within-week placebo, an overlap diagnostic, and the smallest effect this sample could have resolved. On a heavy-tailed outcome the natural-scale mean is carried by a few viral posts; the ratio and the median are the numbers to read.

In [ ]:
from adlift.causal import full_analysis

analysis = full_analysis(timeline, model_factory=lambda: CTRModel(backend=BACKEND, spec=THREADS_SPEC), spec=THREADS_SPEC, n_boot=300)
total = analysis["total_effect"]
print(f"naive:          {analysis['naive_ratio'] * 100:+.1f}% views")
print(f"T-learner:      {total.ratio * 100:+.1f}%  [{total.ratio_low * 100:+.0f}%, {total.ratio_high * 100:+.0f}%]  (headline; S-learner cross-check {analysis['cross_check'].ratio * 100:+.1f}%)")
print(f"planted truth:  {true_log_ate(timeline, 'total') * 100:+.1f}%")
print(f"checks: agree={analysis['estimators_agree']}, placebo x{analysis['placebo']['ratio']:.1f}, overlap AUC {analysis['overlap']['auc']:.2f}, "
      f"detectable ~{analysis['detectable_effect']:,.0f} views on {analysis['n_llm']} LLM / {analysis['n_human']} human posts")
analysis["segments"].round(1)

## The learning curve: when does the ranking become useful?

Posts in time order. The model sees the first *n*, ranks the next ten, and those join the context. This is the cold start as it is actually lived. Where the smoothed line first stays above zero is where the account can start trusting the ranking; compare the hosted TabPFN line with the gradient-boosted stand-in when both are available.

In [ ]:
from adlift.coldstart import learning_curve

curves = {"GBDT + TF-IDF": learning_curve(timeline, CTRModel(backend="baseline", spec=THREADS_SPEC), start=10, step=10)}
if BACKEND != "baseline":
    curves["TabPFN 3.5"] = learning_curve(timeline, CTRModel(backend=BACKEND, spec=THREADS_SPEC), start=10, step=10)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3.4))
for name, curve in curves.items():
    ax.plot(curve["n_context"], curve["spearman"].rolling(3, min_periods=1, center=True).mean(), marker="o", label=name)
ax.axhline(0, color="grey", linestyle=":")
ax.set(xlabel="posts in context", ylabel="Spearman on the next 10 posts (3-block mean)")
ax.legend(frameon=False)
plt.show()

## Take-aways

- **The loop is the product.** Score, suggest, rewrite, post, learn. TabPFN's in-context fit is what makes each step seconds rather than a retraining job, and `fit_with_cache` makes the scoring step cheap enough to run on every draft.
- **Levers beat verdicts.** "Post in the evening, attach an image, keep it under thirty words, do not post it as a reply" is a policy a person can follow tomorrow; "the LLM is 20% worse" is not.
- **Every number carries its check.** Support counts on levers, a placebo and an overlap diagnostic on the author effect, and the smallest resolvable effect for the sample size. On a new account the honest answer for the first few weeks is often "not yet resolvable", and the tool says so.
- **Your account decides.** The synthetic timeline shows the machinery recovering planted effects. Run `adlift threads fetch` on your own account and the same report describes you.